<a href="https://colab.research.google.com/github/LeandroHCarvalho/agentes-2026-2-equipe-agentes_especiais/blob/main/TemplateTrabalho_incompleto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Equipe Agentes Especiais

### Integrantes:
* Beatriz
* Lara
* Leandro



### Nosso problema em uma frase:

* Hoje, quem vai alugar um imóvel precisa entender rapidamente todas as obrigações, prazos e taxas do contrato, mas os documentos possuem linguagem jurídica complexa e requisitos técnicos extensos, o que causa insegurança ao assinar e surpresas financeiras indesejadas durante a locação.

* Nosso problema é ajudar uma pessoa que está alugando um imóvel a identificar informações importantes do contrato e relacioná-las com a legislação aplicável, reduzindo a dificuldade causada pela linguagem jurídica.

### Quem sofre com isso?
Pessoas que precisam alugar uma moradia.

### Como se resolve hoje?
Hoje quem precisa alugar uma moradia, fica a mercê, pois os contratos tem linguagem juridica complexa e falta uma compreensão melhor sobre as legislações vigentes. Quem tem acesso a um advogado consegue se resguardar de alguns abusos, mas a grande maioria das pessoas, simplesmente assinam o contrato mesmo com insegurança, pois precisam de um local para morar.

## PEAS







Elemento	No projeto

---


Performance	Identificar corretamente cláusulas e fundamentos jurídicos relevantes
Environment	Contratos de locação em PDF e base de legislação
Actuators	Resposta/análise apresentada ao usuário
Sensors	PDF, texto extraído e informações fornecidas pelo usuário

Analisando a complexidade do problema que nos xxxxx a resolver, optamos por utilizar o Workflow + Base Vetorial (RAG).

##Porque o RAG?

Antes de falar o porque, o que seria o RAG?

##Retrieval-Augmented Generation **ou** Geração Aumentada por Recuperação.

A parte de Generation utilizando esses fundamentos para produzir uma análise final ainda não está implementada nesse notebook.

Isso é importante: não diga que o código atual já faz a geração final baseada no RAG, porque a etapa mostrada termina no Retrieval.

##Vamos utilizar o RAG porque vamos utilizar as legislações vigentes no brasil a respeito de inquilinato e leis referentes presentes no Código Civil e também no Código de Defesa do Consumidor.

#Mãos na massa!

Vamos ao código, que é o que todo mundo quer ver! 🥰

---



# Instalação das dependências

In [81]:
# ============================================
# Instalação das dependências
# ============================================

!pip install -q openai python-dotenv pydantic tenacity pdfplumber scikit-learn numpy reportlab



---



##O que é uma dependência e o que elas fazem?

explicação do que é dependencia e o que fazem....


---


###Cada biblioteca em nosso projeto possui uma função:

###`openai`




É o SDK utilizado para conversar com uma API compatível com a interface da OpenAI.
> depois nós apontamos o cliente para a Groq.

###`python-dotenv`



Usada para gerenciar variáveis de ambiente e segredos de forma segura. Ela carrega chaves de API (como a OPENAI_API_KEY) a partir de um arquivo oculto chamado .env, evitando que você exponha suas chaves diretamente no código.

###`pydantic`

Biblioteca de validação de dados e estruturação de objetos baseada em tipos do Python. É amplamente utilizada em projetos de IA para garantir que as respostas fornecidas pelos modelos sigam um formato exato (como um JSON bem definido) através de Structured Outputs. (explicar Structured Outputs)

###`tenacity`

Biblioteca usada para implementar mecânicas de tentativa e erro (retry). Muito útil para chamadas de API, pois permite reconectar ou tentar novamente caso haja falhas de rede, taxas limite atingidas (rate limits) ou indisponibilidade temporária do serviço.

Se o professor perguntar:

"Onde vocês usam Tenacity?"

A resposta correta é:

"Nesta versão ela está instalada como dependência prevista para controle de tentativas de chamadas, mas a implementação atual ainda não utiliza um retry explícito."

###`pdfplumber`

Ferramenta para extrair texto, tabelas e dados visuais de arquivos PDF. Muito comum em projetos que precisam ler documentos PDF para alimentar modelos de linguagem ou pipelines de RAG (Retrieval-Augmented Generation).

###`scikit-learn`

que são utilizados no Retrieval.

###`numpy`

Usado principalmente para operações com vetores e ordenação dos resultados.

###`reportlab`

Usado para criar o PDF sintético de teste.

##Configuração da API da Groq

In [82]:
# ============================================
# Configuração da API
# ============================================

import os
import json
import time
import types
import unicodedata

from openai import OpenAI


def obter_chave(nome: str) -> str:
    """
    Lê um segredo dos Secrets do Colab.
    Fora do Colab, utiliza variável de ambiente.
    """
    try:
        from google.colab import userdata
        return userdata.get(nome)

    except ImportError:
        valor = os.getenv(nome)

        if not valor:
            raise RuntimeError(
                f"Defina {nome} nos Secrets do Colab ou no ambiente."
            )

        return valor


LLM_BASE_URL = "https://api.groq.com/openai/v1"

LLM_API_KEY = obter_chave("GROQ_API_KEY")

LLM_MODEL = "openai/gpt-oss-120b"

PRECOS = {
    "openai/gpt-oss-20b": {
        "entrada": 0.075,
        "saida": 0.30
    },
    "openai/gpt-oss-120b": {
        "entrada": 0.150,
        "saida": 0.60
    }
}

TPM = 8_000


cliente = OpenAI(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)


print("✅ Chave carregada, termina em:", LLM_API_KEY[-4:])
print("🤖 Modelo:", LLM_MODEL)
print(f"📊 TPM do plano gratuito: {TPM:,}")

✅ Chave carregada, termina em: WtUg
🤖 Modelo: openai/gpt-oss-120b
📊 TPM do plano gratuito: 8,000


##Comunicação com o LLM

In [83]:
# ============================================
# Função de conversa com o LLM (abstração)
# ============================================

def conversar(
    mensagem_do_usuario: str,
    instrucao_de_sistema: str | None = None,
    temperatura: float = 0.0,
) -> str:
    """
    Envia uma mensagem ao modelo e devolve o texto da resposta.
    """

    mensagens = []

    if instrucao_de_sistema:
        mensagens.append({
            "role": "system",
            "content": instrucao_de_sistema
        })

    mensagens.append({
        "role": "user",
        "content": mensagem_do_usuario
    })

    resposta = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=mensagens,
        temperature=temperatura
    )

    return resposta.choices[0].message.content or ""

##Teste de conexão com a API

In [84]:
# ============================================
# Teste de sanidade da API
# ============================================

print("🔍 Testando conexão com a API...")

resposta_teste = conversar(
    mensagem_do_usuario=(
        "Responda em uma frase: "
        "O que é a Lei do Inquilinato?"
    ),
    instrucao_de_sistema=(
        "Seja conciso e responda em português do Brasil."
    )
)

print("✅ Conexão bem-sucedida!")
print(f"🤖 Resposta do Modelo: {resposta_teste}")

🔍 Testando conexão com a API...
✅ Conexão bem-sucedida!
🤖 Resposta do Modelo: A Lei do Inquilinato (Lei nº 8.245/1991) regula as relações de locação de imóveis urbanos, estabelecendo direitos e deveres de locadores e locatários.


##Modelo estruturado do contrato

In [85]:
# ============================================
# Modelo estruturado do contrato
# ============================================

from pydantic import BaseModel, Field


class DadosContrato(BaseModel):

    valor_aluguel: float = Field(
        description="Valor mensal do aluguel em reais (float)"
    )

    valor_condominio: float = Field(
        description=(
            "Valor estimado do condomínio em reais, "
            "0.0 se não houver"
        )
    )

    valor_iptu: float = Field(
        description=(
            "Valor estimado do IPTU em reais, "
            "0.0 se não houver"
        )
    )

    indice_reajuste: str = Field(
        description=(
            "Índice de reajuste anual citado no contrato "
            "(ex: IGP-M, IPCA)"
        )
    )

    prazo_meses: int = Field(
        description="Prazo total do contrato em meses"
    )

    tipo_garantia: str = Field(
        description="Tipos de garantia exigidos no contrato"
    )

    clausula_reformas: str = Field(
        description=(
            "Resumo da cláusula sobre benfeitorias, "
            "reformas e manutenção"
        )
    )

    clausula_multa_rescisao: str = Field(
        description=(
            "Resumo das regras e valores de multa "
            "por quebra antecipada"
        )
    )

##PDF sintético para teste

In [86]:
# ============================================
# Geração do contrato em PDF para teste
# ============================================

# Melhorar o contrato. Colocar um contrato mais completo

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


def gerar_contrato_sintetico_pdf(
    caminho_arquivo="contrato_exemplo.pdf"
):
    c = canvas.Canvas(
        caminho_arquivo,
        pagesize=letter
    )

    texto_contrato = [
        "CONTRATO DE LOCAÇÃO DE IMÓVEL RESIDENCIAL",
        "",
        "CLÁUSULA 1ª - DO OBJETO: "
        "Locação do imóvel residencial situado na Rua das Flores, 123.",

        "CLÁUSULA 2ª - DO VALOR E REAJUSTE: "
        "O valor do aluguel mensal é de R$ 2.500,00.",

        "O reajuste será anual, fixado exclusivamente pelo índice IGP-M.",

        "CLÁUSULA 3ª - DAS TAXAS: "
        "O locatário pagará Condomínio (R$ 400,00) e IPTU (R$ 100,00).",

        "Parágrafo Único: "
        "O locatário também arcará com o Fundo de Reserva "
        "extraordinário do condomínio.",

        "CLÁUSULA 4ª - DO PRAZO: "
        "A locação terá duração de 30 meses.",

        "CLÁUSULA 5ª - DA GARANTIA: "
        "Para garantia do contrato, exige-se caução em dinheiro "
        "no valor de 3 aluguéis E TAMBÉM a apresentação "
        "de 1 fiador proprietário de imóvel.",

        "CLÁUSULA 6ª - DAS REFORMAS: "
        "Todas as benfeitorias necessárias e reparos estruturais "
        "no imóvel (como vazamentos e rachaduras no telhado) "
        "serão de responsabilidade financeira do locatário.",

        "CLÁUSULA 7ª - DA RESCISÃO: "
        "Em caso de saída antecipada, o locatário pagará "
        "multa fixa de 50% do valor restante total do contrato, "
        "sem proporcionalidade do tempo cumprido."
    ]

    y = 750

    for linha in texto_contrato:

        if "CONTRATO" in linha:
            c.setFont("Helvetica-Bold", 12)
        else:
            c.setFont("Helvetica", 10)

        c.drawString(50, y, linha)

        y -= 20

    c.save()

    print(
        f"📄 Contrato sintético criado com sucesso: "
        f"'{caminho_arquivo}'!"
    )


gerar_contrato_sintetico_pdf()

📄 Contrato sintético criado com sucesso: 'contrato_exemplo.pdf'!


##Leitura do PDF

In [87]:
# ============================================
# Leitura do PDF
# ============================================

import pdfplumber


def ler_pdf(caminho_pdf: str) -> str:
    """
    Extrai o texto bruto do PDF.
    """

    texto_completo = ""

    with pdfplumber.open(caminho_pdf) as pdf:

        for pagina in pdf.pages:

            texto_pagina = pagina.extract_text()

            if texto_pagina:
                texto_completo += texto_pagina + "\n"

    return texto_completo

##Teste de leitura do PDF

In [88]:
# Testando leitura do PDF

texto_bruto = ler_pdf("contrato_exemplo.pdf")

print("📄 Texto extraído:")
print(texto_bruto)

📄 Texto extraído:
CONTRATO DE LOCAÇÃO DE IMÓVEL RESIDENCIAL
CLÁUSULA 1ª - DO OBJETO: Locação do imóvel residencial situado na Rua das Flores, 123.
CLÁUSULA 2ª - DO VALOR E REAJUSTE: O valor do aluguel mensal é de R$ 2.500,00.
O reajuste será anual, fixado exclusivamente pelo índice IGP-M.
CLÁUSULA 3ª - DAS TAXAS: O locatário pagará Condomínio (R$ 400,00) e IPTU (R$ 100,00).
Parágrafo Único: O locatário também arcará com o Fundo de Reserva extraordinário do condomínio.
CLÁUSULA 4ª - DO PRAZO: A locação terá duração de 30 meses.
CLÁUSULA 5ª - DA GARANTIA: Para garantia do contrato, exige-se caução em dinheiro no valor de 3 aluguéis E TAMBÉM a apresentação de 1 fiador proprietário de imóvel.
CLÁUSULA 6ª - DAS REFORMAS: Todas as benfeitorias necessárias e reparos estruturais no imóvel (como vazamentos e rachaduras no telhado) serão de responsabilidade financeira do locatário.
CLÁUSULA 7ª - DA RESCISÃO: Em caso de saída antecipada, o locatário pagará multa fixa de 50% do valor restante tota

##Extração estruturada usando o LLM

In [89]:
# ============================================
# Extração estruturada
# ============================================

def extrair_dados_estruturados(
    texto_contrato: str
) -> DadosContrato:
    """
    Utiliza o LLM para estruturar os dados do contrato.
    """

    instrucao = (
        "Você é um extrator especialista em contratos imobiliários. "
        "Analise o texto fornecido e extraia com precisão "
        "os campos solicitados."
    )

    resposta = cliente.beta.chat.completions.parse(
        model=LLM_MODEL,

        messages=[
            {
                "role": "system",
                "content": instrucao
            },
            {
                "role": "user",
                "content": texto_contrato
            }
        ],

        response_format=DadosContrato,

        temperature=0.0
    )

    return resposta.choices[0].message.parsed

##Executar a extração

In [90]:
# ============================================
# Execução da extração
# ============================================

print("1. 📄 Lendo o PDF...")

texto_bruto = ler_pdf(
    "contrato_exemplo.pdf"
)


print("2. 🤖 Extraindo dados estruturados via LLM...")

dados_extraidos = extrair_dados_estruturados(
    texto_bruto
)


print("\n✅ Extração Estruturada Concluída!\n")

print(
    f"• Aluguel: "
    f"R$ {dados_extraidos.valor_aluguel:.2f}"
)

print(
    f"• Condomínio: "
    f"R$ {dados_extraidos.valor_condominio:.2f}"
)

print(
    f"• IPTU: "
    f"R$ {dados_extraidos.valor_iptu:.2f}"
)

print(
    f"• Reajuste: "
    f"{dados_extraidos.indice_reajuste}"
)

print(
    f"• Prazo: "
    f"{dados_extraidos.prazo_meses} meses"
)

print(
    f"• Garantia Exigida: "
    f"{dados_extraidos.tipo_garantia}"
)

print(
    f"• Reformas: "
    f"{dados_extraidos.clausula_reformas}"
)

print(
    f"• Multa Rescisão: "
    f"{dados_extraidos.clausula_multa_rescisao}"
)

1. 📄 Lendo o PDF...
2. 🤖 Extraindo dados estruturados via LLM...

✅ Extração Estruturada Concluída!

• Aluguel: R$ 2500.00
• Condomínio: R$ 400.00
• IPTU: R$ 100.00
• Reajuste: IGP-M
• Prazo: 30 meses
• Garantia Exigida: caução em dinheiro equivalente a 3 aluguéis e apresentação de 1 fiador proprietário de imóvel
• Reformas: Todas as benfeitorias necessárias e reparos estruturais no imóvel (como vazamentos e rachaduras no telhado) serão de responsabilidade financeira do locatário.
• Multa Rescisão: Multa fixa de 50% do valor restante total do contrato, sem proporcionalidade do tempo cumprido.


##Base jurídica

In [91]:
# ============================================
# Base de conhecimento jurídica
# ============================================

#Ampliar a base de dados******

LEIS_E_JURISPRUDENCIAS = [

    {
        "id": "art_37",

        "topico": "Garantia Locatícia / Dupla Garantia",

        "texto": (
            "Art. 37 da Lei 8.245/91: "
            "No contrato de locação, pode o locador exigir "
            "as seguintes modalidades de garantia: "
            "I - caução; II - fiança; "
            "III - seguro de fiança locatícia. "
            "Parágrafo único: É vedada, sob pena de nulidade, "
            "mais de uma das modalidades de garantia "
            "num mesmo contrato de locação."
        )
    },

    {
        "id": "art_22_fundo_reserva",

        "topico": (
            "Despesas Extraordinárias e Fundo de Reserva"
        ),

        "texto": (
            "Art. 22 da Lei 8.245/91: "
            "O locador (proprietário) é obrigado a responder "
            "pelas despesas extraordinárias de condomínio, "
            "as quais incluem reformas estruturais, pintura "
            "de fachada e constituição do fundo de reserva."
        )
    },

    {
        "id": "art_22_reformas_estruturais",

        "topico": "Obras e Reparos Estruturais",

        "texto": (
            "Art. 22, X da Lei 8.245/91: "
            "O locador é obrigado a pagar as obras de reformas "
            "ou acréscimos que interessem à estrutura integral "
            "do imóvel, vazamentos estruturais e vícios "
            "anteriores à locação."
        )
    },

    {
        "id": "art_4_multa_proporcional",

        "topico": (
            "Multa por Rescisão Antecipada e Proporcionalidade"
        ),

        "texto": (
            "Art. 4º da Lei 8.245/91: "
            "Durante o prazo estipulado para a duração do contrato, "
            "não poderá o locador reaver o imóvel alugado. "
            "O locatário, todavia, poderá devolvê-lo, "
            "pagando a multa pactuada, proporcional ao período "
            "de cumprimento do contrato."
        )
    },

    {
        "id": "art_18_reajuste_indice",

        "topico": "Índices de Reajuste do Aluguel",

        "texto": (
            "Art. 18 da Lei 8.245/91: "
            "É lícito às partes fixar de comum acordo novo valor "
            "para o aluguel, bem como inserir ou modificar "
            "cláusula de reajuste. É vedada a fixação do valor "
            "em moeda estrangeira e a vinculação à variação "
            "do salário mínimo."
        )
    }
]

##Criar o índice TF-IDF

###O que é TF-IDF?

**TF-IDF significa:**

Term Frequency — Inverse Document Frequency

**O que ele faz?**

Ele transforma textos em representação numérica.
A ideia intuitiva é que palavras importantes para um documento recebem maior peso, enquanto palavras muito comuns entre vários documentos recebem menor peso.

**Por exemplo:**

`caução - fiador - garantia` **
serão relevantes para o documento (presentes no Art. 37).

Enquanto palavras muito genéricas terão menos poder de diferenciação.

In [92]:
# ============================================
# Indexação da base jurídica
# ============================================

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


corpus_textos = [
    documento["texto"]
    for documento in LEIS_E_JURISPRUDENCIAS
]


vectorizer = TfidfVectorizer()


print(
    "🔍 Gerando matriz vetorial "
    "para os artigos da base jurídica..."
)


vetores_leis = vectorizer.fit_transform(
    corpus_textos
)


print(
    f"✅ Base legal indexada! "
    f"Total de {len(LEIS_E_JURISPRUDENCIAS)} "
    f"artigos convertidos em vetores."
)

🔍 Gerando matriz vetorial para os artigos da base jurídica...
✅ Base legal indexada! Total de 5 artigos convertidos em vetores.


##Função correta de Retrieval

In [93]:
# ============================================
# Retrieval do RAG
# ============================================

def buscar_fundamento_legal(
    query: str,
    top_k: int = 4,
    limite_similaridade: float = 0.10
) -> list[dict]:
    """
    Busca na base jurídica os artigos mais
    relevantes para a consulta utilizando
    TF-IDF + similaridade de cosseno.
    """

    # 1. Transforma a consulta em vetor
    vetor_query = vectorizer.transform(
        [query]
    )

    # 2. Calcula a similaridade entre
    #    consulta e todos os documentos
    similaridades = cosine_similarity(
        vetor_query,
        vetores_leis
    )[0]

    # 3. Ordena do maior para o menor score
    indices_top_k = np.argsort(
        similaridades
    )[::-1][:top_k]

    # 4. Monta os resultados
    resultados = []

    for idx in indices_top_k:

        score = float(similaridades[idx])

        if score < limite_similaridade:
            continue

        resultados.append({
            "artigo": LEIS_E_JURISPRUDENCIAS[idx],
            "score_similaridade": float(
                similaridades[idx]
            )
        })

        if len(resultados) >= top_k:
          break

    return resultados

##Testar o Retrieval

In [94]:
# ============================================
# Teste do Retrieval
# ============================================

print(
    "\n🔍 Testando o RAG:"
)

print(
    "Buscando leis aplicáveis sobre "
    "'exigir caução e fiador ao mesmo tempo'...\n"
)


busca_teste = buscar_fundamento_legal(
    "O contrato pede pagamento de caução e também fiador",
    top_k=4
)


for i, item in enumerate(
    busca_teste,
    start=1
):

    print(
        f"--- Resultado {i} "
        f"(Similaridade: "
        f"{item['score_similaridade']:.4f}) ---"
    )

    print(
        f"📌 Tópico: "
        f"{item['artigo']['topico']}"
    )

    print(
        f"📜 Texto Legal: "
        f"{item['artigo']['texto']}\n"
    )


🔍 Testando o RAG:
Buscando leis aplicáveis sobre 'exigir caução e fiador ao mesmo tempo'...

--- Resultado 1 (Similaridade: 0.3981) ---
📌 Tópico: Garantia Locatícia / Dupla Garantia
📜 Texto Legal: Art. 37 da Lei 8.245/91: No contrato de locação, pode o locador exigir as seguintes modalidades de garantia: I - caução; II - fiança; III - seguro de fiança locatícia. Parágrafo único: É vedada, sob pena de nulidade, mais de uma das modalidades de garantia num mesmo contrato de locação.

--- Resultado 2 (Similaridade: 0.2042) ---
📌 Tópico: Multa por Rescisão Antecipada e Proporcionalidade
📜 Texto Legal: Art. 4º da Lei 8.245/91: Durante o prazo estipulado para a duração do contrato, não poderá o locador reaver o imóvel alugado. O locatário, todavia, poderá devolvê-lo, pagando a multa pactuada, proporcional ao período de cumprimento do contrato.

--- Resultado 3 (Similaridade: 0.1115) ---
📌 Tópico: Despesas Extraordinárias e Fundo de Reserva
📜 Texto Legal: Art. 22 da Lei 8.245/91: O locador (p

% de confiabilidade do contrato em relação à legislação... o quanto confiavel o contrato é em relação a legislação.

In [95]:
from pydantic import BaseModel, Field

# 1. Modelo Pydantic para estruturar a análise de cada cláusula
class AvaliacaoClausula(BaseModel):
    clausula_nome: str = Field(description="Nome da cláusula analisada")
    status: str = Field(description="Classificação: 'CONFORME', 'ATENÇÃO' ou 'ILEGAL'")
    pontuacao: float = Field(description="Pontuação de 0.0 (totalmente ilegal) a 1.0 (totalmente legal)")
    justificativa_simples: str = Field(description="Explicação em linguagem simples e direta para o leigo")
    fundamento_legal: str = Field(description="Citação do artigo de lei recuperado via RAG")

class RelatorioAuditoria(BaseModel):
    avaliacoes: list[AvaliacaoClausula]
    resumo_executivo: str = Field(description="Resumo geral dos riscos encontrados no contrato")

# 2. Função principal para auditoria automatizada
def auditar_contrato(dados: DadosContrato) -> dict:

    # Pontos críticos a serem auditados contra a Lei do Inquilinato
    itens_para_auditar = [
        {"nome": "Exigência de Garantias", "texto": dados.tipo_garantia, "busca": "dupla garantia caução fiador cumulação vedada"},
        {"nome": "Reformas e Obras Estruturais", "texto": dados.clausula_reformas, "busca": "obras reformas estruturais vazamento responsabilidade locador"},
        {"nome": "Despesas Condominiais e Fundo de Reserva", "texto": f"Condomínio R$ {dados.valor_condominio}", "busca": "fundo de reserva despesas extraordinarias locador"},
        {"nome": "Multa por Rescisão Antecipada", "texto": dados.clausula_multa_rescisao, "busca": "multa rescisao antecipada proporcionalidade tempo cumprido"},
        {"nome": "Índice de Reajuste", "texto": dados.indice_reajuste, "busca": "indice reajuste IGPM IPCA reajuste anual"}
    ]

    avaliacoes_resultados = []

    for item in itens_para_auditar:
        # RAG Retrieval: Busca os artigos de lei mais próximos
        leis_relevantes = buscar_fundamento_legal(item["busca"], top_k=2)
        contexto_legal = "\n".join([f"- {l['artigo']['texto']}" for l in leis_relevantes])

        prompt_sistema = (
            "Você é um auditor jurídico especialista na Lei do Inquilinato (Lei 8.245/91).\n"
            "Analise a cláusula contratual em relação aos artigos da lei fornecidos.\n\n"
            "Regras de Pontuação:\n"
            "- Se for totalmente legal/conforme: status='CONFORME', pontuacao=1.0\n"
            "- Se for ambígua ou desfavorável sem ser explicitamente nula: status='ATENÇÃO', pontuacao=0.5\n"
            "- Se violar abertamente a lei (cláusula nula): status='ILEGAL', pontuacao=0.0\n\n"
            "Traduza a justificativa para linguagem simples e direta, sem 'juridiquês'."
        )

        prompt_usuario = (
            f"Cláusula/Item: {item['nome']}\n"
            f"Texto no Contrato: {item['texto']}\n\n"
            f"Base Legal Recuperada (RAG):\n{contexto_legal}"
        )

        resposta = cliente.beta.chat.completions.parse(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": prompt_usuario}
            ],
            response_format=AvaliacaoClausula,
            temperature=0.0
        )

        avaliacoes_resultados.append(resposta.choices[0].message.parsed)

    # Cálculo do Índice de Confiabilidade do Contrato
    soma_pontos = sum([a.pontuacao for a in avaliacoes_resultados])
    total_itens = len(avaliacoes_resultados)
    porcentagem_confiabilidade = (soma_pontos / total_itens) * 100

    return {
        "porcentagem_confiabilidade": porcentagem_confiabilidade,
        "avaliacoes": avaliacoes_resultados
    }

# 3. Executando a Auditoria nos dados da Etapa 2
print("⏳ Executando auditoria legal e calculando índice de confiabilidade...\n")
resultado_auditoria = auditar_contrato(dados_extraidos)

# 4. Exibição do Painel de Resultados
score = resultado_auditoria["porcentagem_confiabilidade"]

print("=" * 60)
print(f"📊 ÍNDICE DE CONFIABILIDADE DO CONTRATO: {score:.1f}%")
if score >= 80:
    print("🟢 Nível de Risco: BAIXO (Contrato equilibrado)")
elif score >= 50:
    print("🟡 Nível de Risco: MÉDIO (Requer ajustes e atenção)")
else:
    print("🔴 Nível de Risco: ALTO (Possui cláusulas abusivas ou nulas)")
print("=" * 60 + "\n")

print("📋 DETALHAMENTO DA AUDITORIA POR CLÁUSULA:\n")
for idx, item in enumerate(resultado_auditoria["avaliacoes"], 1):
    icone = "✅" if item.status == "CONFORME" else ("⚠️" if item.status == "ATENÇÃO" else "❌")
    print(f"{idx}. {icone} [{item.status}] - {item.clausula_nome}")
    print(f"   • Explicando em linguagem simples: {item.justificativa_simples}")
    print(f"   • Base Legal: {item.fundamento_legal}\n")

⏳ Executando auditoria legal e calculando índice de confiabilidade...

📊 ÍNDICE DE CONFIABILIDADE DO CONTRATO: 30.0%
🔴 Nível de Risco: ALTO (Possui cláusulas abusivas ou nulas)

📋 DETALHAMENTO DA AUDITORIA POR CLÁUSULA:

1. ❌ [ILEGAL] - Exigência de Garantias
   • Explicando em linguagem simples: A lei só permite escolher uma única forma de garantia (caução ou fiador). Pedir as duas ao mesmo tempo é proibido, então a cláusula não vale.
   • Base Legal: Art. 37 da Lei 8.245/91, parágrafo único: é vedada, sob pena de nulidade, mais de uma das modalidades de garantia num mesmo contrato de locação.

2. ❌ [ILEGAL] - Reformas e Obras Estruturais
   • Explicando em linguagem simples: A lei deixa claro que quem cuida de consertos estruturais, como vazamentos e rachaduras, é o proprietário, não o inquilino. Portanto, a cláusula que coloca essa obrigação no locatário vai contra a lei e não tem validade.
   • Base Legal: Art. 22, X da Lei 8.245/91 e Art. 22 da Lei 8.245/91 – cabem ao locador as d